In [31]:
# Move-one-unit: move 1 unit from one part to another.
# If a part of size 1 loses its unit, that part is removed.
# A unit can also move into a brand-new part of size 1.
def neighborsMove(partition):
  parts = list(partition)
  results = set()
  n = sum(parts)
  for i in range(len(parts)):
    for j in range(len(parts)):
      if i == j:
        continue
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts[j] += 1
      new_parts = [p for p in new_parts if p > 0]
      new_parts_sorted = tuple(sorted(new_parts, reverse=True))
      if new_parts_sorted and sum(new_parts_sorted) == n:
        results.add(new_parts_sorted)
    # Split off a new part of size 1 from part i
    if parts[i] > 1:
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts.append(1)
      results.add(tuple(sorted(new_parts, reverse=True)))
  return results - {partition}

# Split/merge: split one part into two positive parts, or merge two parts into one.
def neighborsSplitMerge(partition):
  parts = list(partition)
  results = set()
  # Splits
  for i in range(len(parts)):
    for a in range(1, parts[i]):
      b = parts[i] - a
      new_parts = parts[:i] + parts[i+1:] + [a, b]
      results.add(tuple(sorted(new_parts, reverse=True)))
  # Merges
  for i in range(len(parts)):
    for j in range(i+1, len(parts)):
      merged = parts[i] + parts[j]
      new_parts = [parts[k] for k in range(len(parts)) if k != i and k != j] + [merged]
      results.add(tuple(sorted(new_parts, reverse=True)))
  return results - {partition}

# Combined neighborhood: union of split/merge and move-one-unit neighbors.
def neighborsCombined(partition):
  return neighborsSplitMerge(partition) | neighborsMove(partition)


In [32]:
def toBinary(partition):
    return ''.join('0' * (p - 1) + '1' for p in partition)

In [33]:
# Warnsdorff's rule on the combined neighborhood, starting from (1,...,1).
#
# At each step, choose the unvisited neighbor with the FEWEST unvisited neighbors
# of its own. Break ties using lexicographic order.
#
# Intuition: visiting low-degree nodes first prevents them from becoming
# stranded (unreachable) later in the traversal.
def greedyWarnsdorff(n):
  visited = set()
  word = (1,) * n
  visited.add(word)
  yield word
  while True:
    candidates = [c for c in neighborsCombined(word) if c not in visited]
    if not candidates:
      break
    word = min(candidates, key=lambda c: (
        len([x for x in neighborsCombined(c) if x not in visited]),
        c
    ))
    visited.add(word)
    yield word

In [34]:
# Classify the operation from partition p to partition q.
# Returns one of: 'move', 'split', 'merge', 'unknown'.
def classifyOp(p, q):
    if q in neighborsSplitMerge(p):
        return 'split' if len(q) > len(p) else 'merge'
    if q in neighborsMove(p):
        return 'move'
    return 'unknown'

# Extract the annotated operation sequence for a given n.
def operationSequence(n):
    path = list(greedyWarnsdorff(n))
    steps = []
    for i in range(len(path) - 1):
        p, q = path[i], path[i + 1]
        steps.append({
            'from': p,
            'to': q,
            'op': classifyOp(p, q),
            'from_parts': len(p),
            'to_parts': len(q),
            'from_max': p[0],
            'to_max': q[0],
        })
    return steps

In [43]:
def ferrersDiagram(partition):
    diagram = []
    for p in partition:
        row = ['.'] * p
        diagram.append(''.join(row))
    return '\n'.join(diagram)

In [48]:
n = 6
seen = []
for p in greedyWarnsdorff(n):
    if p in seen:
        print("Already seen:", p)
    else:
        print(classifyOp(seen[-1], p) if len(seen)>0 else '', "\n", ferrersDiagram(p), "\n", p, "\n", sep='')
        seen.append(p)

print(len(seen), "partitions generated.")


.
.
.
.
.
.
(1, 1, 1, 1, 1, 1)

merge
..
.
.
.
.
(2, 1, 1, 1, 1)

merge
...
.
.
.
(3, 1, 1, 1)

move
..
..
.
.
(2, 2, 1, 1)

merge
..
..
..
(2, 2, 2)

move
...
..
.
(3, 2, 1)

merge
...
...
(3, 3)

merge
......
(6,)

split
....
..
(4, 2)

split
....
.
.
(4, 1, 1)

merge
.....
.
(5, 1)

11 partitions generated.
